# Automatic Planning Pattern Demo - Planner Plus ReAct Loop


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f643b6891-84f6-4672-aa1f-4724c5ad2d12_716x526-3.gif" alt="Planning pattern" width="500"/>

This notebook is the automatic demo companion to `03b_lab_planner_react_workflow.ipynb`.

Use it when you want to see a more realistic bounded loop where:
- `PlanningAgent` chooses the next evidence-gathering step
- `ReactAgent` carries out that step with tools
- the returned `observation` is fed back into replanning automatically

This notebook is mainly for instructor demo or optional advanced exploration. It may be slower and more variable than the guided notebook because the loop depends on live model behavior.


## What You Will Do

This notebook runs a bounded automatic loop.

1. Load the Lab 4 setup and tool definitions.
2. Let `PlanningAgent` build the first plan.
3. Let `PlanningAgent` assign the next task to `ReactAgent`.
4. Let `ReactAgent` gather evidence and return an `observation`.
5. Feed that observation back to `PlanningAgent` automatically.
6. Repeat for up to `5` rounds.
7. Stop early if `PlanningAgent` returns `FINISHED:` or the loop gets stuck.
8. Review the round-by-round summaries and final report.


## Lab Question and Response Format

Use the staged artifacts to answer this planning question:

**What timeline best explains the missing-phone interval, what remaining dependency or uncertainty does the first ReactAgent result identify, and how should the explanation change after the next evidence check?**

The planner-control helper returns one of two short formats:
- `NEXT_TASK: ...` means `PlanningAgent` wants `ReactAgent` to gather one more piece of evidence.
- `FINISHED: ...` means `PlanningAgent` thinks the evidence is sufficient and the workflow can stop.

This notebook keeps the same six local tools visible from the start. The loop is capped at `5` rounds so the demo stays bounded for classroom use.


### Step 1: Set Up the Notebook

Run this setup cell first. It prepares the notebook environment before any planning or tool use happens.

In plain language, this cell does four things:
- loads settings from `.env`
- checks that you opened the notebook from the correct Lab 4 folder
- imports the agent classes and helper libraries
- creates the client and the path to the staged case data

A few names to notice:
- `MODEL` is the local model name from `.env`
- `OLLAMA_BASE_URL` tells the notebook where the local model server is running
- `client` is the object used to send requests to that model server
- `data_dir` is the folder that contains the Lab 4 evidence files


In [ ]:

import csv
import json
import os
import string
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and prepares the case data.
LAB_NAME = 'lab4_planning_pattern'

# `lab_dir` is the folder where this notebook is currently running.
lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

# Move one level up so the notebook can also find the shared `src/` code.
repo_root = lab_dir.parent

# Check that the template settings file exists.
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

# Check that the real local settings file exists too.
env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

# Add the shared project code to Python's import path so notebook imports will work.
src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

# Load model settings from the lab-local `.env` file.
load_dotenv(env_path, override=True)

# Read the model name and local server URL from the environment.
MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

# `data_dir` is the folder that stores the staged evidence files for this lab.
data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Could not find the Lab 4 data folder')

# Import the packaged agents and the `@tool` wrapper used in this course.
from agentic_patterns.planning_pattern.planning_agent import PlanningAgent
from agentic_patterns.react_pattern.react_agent import ReactAgent
from agentic_patterns.tool_pattern.tool import tool

# Create the OpenAI-compatible client that will talk to the local model server.
client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

# Print a few values so students can confirm the notebook is pointed at the right place.
print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print('Data files:', sorted(path.name for path in data_dir.iterdir() if path.is_file()))


### Step 2: Define the `ReactAgent` Tools

These are the same local evidence tools used in the guided notebook. They keep the automatic demo grounded in the same case data and tool menu.

For non-CS students, a `tool` here is just a small Python helper function that opens one part of the evidence and returns a readable result.

Examples:
- `get_call_log()` returns the phone-call records from `call_log.csv`
- `search_network_evidence('offline')` looks through the local Lab 4 files for network-related clues

Notice the `@tool` line above each function. That wrapper makes the function visible to `ReactAgent`, so the model can choose and use it during the loop.

At the end of the cell, `tool_schema_preview` shows the tool signatures in a machine-readable form. This is what helps the model know each tool's name and inputs.


In [ ]:

# Helper function: read one CSV evidence file and return its rows.
def read_csv(filename: str) -> list[dict]:
    # `csv.DictReader` turns each CSV row into a Python dictionary.
    with (data_dir / filename).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


# Helper function: return one JSON string with indentation for readability.
def as_pretty_json(payload: dict | list) -> str:
    return json.dumps(payload, indent=2)


# Tool 1: show which evidence files are available in this staged case package.
@tool
def list_artifact_files() -> str:
    """Return the artifact files available in the Lab 4 data folder."""
    artifact_files = sorted(path.name for path in data_dir.iterdir() if path.is_file())
    return as_pretty_json({'artifact_files': artifact_files})


# Tool 2: return the main incident window from the case manifest.
@tool
def get_incident_window() -> str:
    """Return the UTC start and end timestamps for the missing-phone interval."""
    manifest = json.loads((data_dir / 'artifact_manifest.json').read_text(encoding='utf-8'))
    incident_window = manifest['incident_window_utc']
    return as_pretty_json(
        {
            'start': incident_window['start'],
            'end': incident_window['end'],
            'analysis_timezone': manifest['analysis_timezone'],
        }
    )


# Tool 3: return the unlock and lock events that bound device access in the case.
@tool
def get_unlock_events() -> str:
    """Return the recorded unlock and lock events for the device."""
    rows = read_csv('unlock_events.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 4: return the phone call log summary.
@tool
def get_call_log() -> str:
    """Return the recorded phone call events for the incident period."""
    rows = read_csv('call_log.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 5: return the WhatsApp activity summary.
@tool
def get_whatsapp_events() -> str:
    """Return the WhatsApp activity relevant to the incident period."""
    rows = read_csv('whatsapp_events.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 6: search the local evidence files for network-related clues.
@tool
def search_network_evidence(query: str) -> str:
    """Search the local Lab 4 data folder for network-related evidence that matches the query."""
    # Break the query into lowercase search terms such as "offline" or "network".
    query_terms = {
        token.strip(string.punctuation).lower()
        for token in query.split()
        if token.strip(string.punctuation)
    }
    if not query_terms:
        query_terms = {'network'}

    matches = []
    # Walk through each local evidence file and keep a few matching lines.
    for path in sorted(data_dir.iterdir()):
        if not path.is_file():
            continue
        file_matches = []
        for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
            lower_line = line.lower()
            if any(term in lower_line for term in query_terms):
                file_matches.append({'line_number': line_number, 'text': line})
            # Limit the number of matches per file so the output stays readable.
            if len(file_matches) >= 4:
                break
        if file_matches:
            matches.append({'file': path.name, 'matches': file_matches})

    # If there are no exact matches, still show the beginning of `network_status.csv`
    # so students can see where connectivity evidence lives in the case package.
    if not matches:
        fallback_lines = (data_dir / 'network_status.csv').read_text(encoding='utf-8').splitlines()[:4]
        return as_pretty_json(
            {
                'query': query,
                'matches': [],
                'note': 'No exact keyword match found. Showing the start of network_status.csv as a fallback.',
                'fallback_excerpt': fallback_lines,
            }
        )

    return as_pretty_json({'query': query, 'matches': matches})


# All six tools are visible from the start in this teaching version.
react_tools = [
    list_artifact_files,
    get_incident_window,
    get_unlock_events,
    get_call_log,
    get_whatsapp_events,
    search_network_evidence,
]

# This preview lets students inspect the machine-readable tool signatures.
tool_schema_preview = {
    tool.name: json.loads(tool.fn_signature)
    for tool in react_tools
}

tool_schema_preview


### Step 3: Define the Planner Control Helper and Automatic Loop

This cell does two jobs:
- it reuses the same `PlanningAgent` control pattern from the guided notebook
- it defines the bounded automatic loop helper that will run up to `5` rounds and store round summaries plus raw `ReactAgent` traces

For non-CS students, the main idea is:
1. `PlanningAgent` looks at the current plan and observations.
2. It returns either `NEXT_TASK: ...` or `FINISHED: ...`.
3. If there is a next task, `ReactAgent` uses tools to gather more evidence.
4. That evidence becomes the next `observation`.
5. The loop repeats until enough evidence has been gathered or the loop hits the safety limit.

The automatic demo rebuilds fresh planning state so it does not depend on any variables from `03b_lab_planner_react_workflow.ipynb`.


In [ ]:
PLANNER_CONTROLLER_SYSTEM_PROMPT = """
You coordinate a planning workflow for a digital forensics case.

Your job is to choose the next single evidence-gathering task for ReactAgent.

Return exactly one of these two formats:
NEXT_TASK: <one concrete task sentence>
FINISHED: <brief reason and final reporting instruction>

Rules:
- Ask for only one task at a time.
- The task must be solvable with the listed ReactAgent tools.
- Start with the most direct timeline-building evidence before broader search.
- Use network-related search only after direct evidence reveals a remaining dependency or uncertainty about connectivity, delivery timing, or online/offline status.
- Keep the task short and specific.
""".strip()


# Ask PlanningAgent for the next task using the current plan and current observations.
def ask_planner_for_next_task(
    case_question: str,
    current_plan: str,
    observations: str,
    tool_names: list[str],
) -> str:
    # If there are no observations yet, tell the planner that explicitly.
    observation_text = observations if observations else 'No ReactAgent observations yet.'
    # Build one prompt that gives the planner the question, current plan, observations, and tools.
    prompt = (
        f'Case question:\n{case_question}\n\n'
        f'Current plan:\n{current_plan}\n\n'
        f'Observations so far:\n{observation_text}\n\n'
        f'Available ReactAgent tools:\n- ' + '\n- '.join(tool_names)
    )
    return client.chat.completions.create(
        messages=[
            {'role': 'system', 'content': PLANNER_CONTROLLER_SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        model=MODEL,
    ).choices[0].message.content.strip()


# Pull the task text out of a PlanningAgent response such as "NEXT_TASK: ...".
def extract_next_task(planner_output: str) -> str | None:
    if 'NEXT_TASK:' in planner_output:
        return planner_output.split('NEXT_TASK:', 1)[1].strip()
    if planner_output.startswith('FINISHED:'):
        return None
    return planner_output.strip()


from contextlib import redirect_stdout
from io import StringIO

# Safety cap: the automatic classroom demo will stop after this many rounds.
AUTO_MAX_ROUNDS = 5

# Keep the main case question in one variable so later helper functions can reuse it.
AUTO_CASE_QUESTION = (
    'What timeline best explains the missing-phone interval, what remaining dependency or uncertainty does the first ReactAgent result identify, '
    'and how should the explanation change after the next evidence check?'
)


# Helper function: rebuild the case question and artifact list for the automatic demo.
def build_automatic_case_context() -> tuple[str, str]:
    artifact_menu = {
        'artifact_files': sorted(path.name for path in data_dir.iterdir() if path.is_file()),
        'note': 'PlanningAgent should choose what ReactAgent needs to inspect next from the full visible tool set.',
    }
    planner_case_context = (
        f'{AUTO_CASE_QUESTION}\n\n'
        f'High-level artifact list:\n{json.dumps(artifact_menu, indent=2)}'
    )
    return AUTO_CASE_QUESTION, planner_case_context


# Helper function: keep the automatic-round instructions consistent from one round to the next.
def build_automatic_round_prompt(task: str) -> str:
    return f"""
Carry out this PlanningAgent-assigned task using the available tools:

{task}

Requirements:
- Carry out only this assigned task.
- Use the available tool or tools that best match the task.
- Mention the important file names, timestamps, and status details you find.
- Return a short response with these labels:
  Evidence gathered:
  Impact on the plan:
""".strip()


# Helper function: normalize task text so repeated-task detection is less brittle.
def normalize_task(task: str) -> str:
    return ' '.join(task.split())


# Helper function: turn the collected round results into one observation block for PlanningAgent.
def format_cumulative_observations(round_logs: list[dict]) -> str:
    if not round_logs:
        return 'No ReactAgent observations were collected.'
    return '\n\n'.join(
        [
            f"ReactAgent round {log['round_number']} result:\n{log['react_response']}"
            for log in round_logs
        ]
    )


# Helper function: build the final-report prompt from the latest plan and all collected observations.
def build_automatic_final_report_prompt(
    case_question: str,
    current_plan: str,
    round_logs: list[dict],
    stop_reason: str,
) -> str:
    combined_observations = format_cumulative_observations(round_logs)
    return (
        'Use the case question, the current investigation plan, and the ReactAgent observations below to write the final forensic report.\n\n'
        'Return a report with:\n'
        '1. Planner step log\n'
        '2. Direct-evidence findings\n'
        '3. Network evidence findings\n'
        '4. Final timeline and evidence-bounded conclusion\n\n'
        f'Loop stop reason:\n{stop_reason}\n\n'
        f'Case question:\n{case_question}\n\n'
        f'Current plan:\n{current_plan}\n\n'
        f'ReactAgent observations:\n{combined_observations}'
    )


# Main helper: run the planning loop automatically for a bounded number of rounds.
def run_automatic_planner_react_demo(max_rounds: int = AUTO_MAX_ROUNDS) -> dict:
    # Start from fresh planning state so this demo does not depend on earlier notebook runs.
    case_question, planner_case_context = build_automatic_case_context()
    planning_agent = PlanningAgent(client=client, model=MODEL)
    initial_plan = planning_agent.build_initial_plan(planner_case_context)
    current_plan = initial_plan

    # `round_logs` will store a readable record of each loop iteration.
    round_logs = []
    previous_task = None
    last_planner_decision = ''
    stop_reason = ''

    for round_number in range(1, max_rounds + 1):
        # Ask PlanningAgent what ReactAgent should do next.
        last_planner_decision = ask_planner_for_next_task(
            case_question=case_question,
            current_plan=current_plan,
            observations=format_cumulative_observations(round_logs),
            tool_names=[tool.name for tool in react_tools],
        )

        # Stop early if the planner says the workflow is finished.
        if last_planner_decision.startswith('FINISHED:'):
            stop_reason = (
                f'PlanningAgent returned FINISHED after {round_number - 1} completed round(s).'
            )
            break

        next_task = extract_next_task(last_planner_decision)
        # Stop if the planner output does not contain a usable task.
        if not next_task:
            stop_reason = (
                f'PlanningAgent did not return a usable NEXT_TASK after {round_number - 1} completed round(s).'
            )
            break

        normalized_task = normalize_task(next_task)
        # Stop if the same task repeats, which usually means the loop is stuck.
        if previous_task is not None and normalized_task == previous_task:
            stop_reason = (
                f'PlanningAgent repeated the same task in round {round_number}, so the demo stopped early to avoid a stuck loop.'
            )
            break

        # Create a fresh ReactAgent each round so the system prompt does not accumulate repeatedly.
        round_react_agent = ReactAgent(tools=react_tools, client=client, model=MODEL)
        trace_buffer = StringIO()
        # Capture the raw printed ReAct trace without flooding the notebook output.
        with redirect_stdout(trace_buffer):
            react_response = round_react_agent.run(
                user_msg=build_automatic_round_prompt(next_task)
            )

        # Build one combined observation string so PlanningAgent can revise the plan.
        combined_observations = format_cumulative_observations(
            round_logs
            + [
                {
                    'round_number': round_number,
                    'react_response': react_response,
                }
            ]
        )
        revised_plan = planning_agent.revise_plan(
            user_msg=planner_case_context,
            current_plan=current_plan,
            observations=combined_observations,
        )

        # Store the main artifacts from this round in a structured log.
        round_logs.append(
            {
                'round_number': round_number,
                'planner_decision': last_planner_decision,
                'task': next_task,
                'react_response': react_response,
                'revised_plan': revised_plan,
                'react_trace': trace_buffer.getvalue(),
            }
        )
        current_plan = revised_plan
        previous_task = normalized_task
    else:
        # This branch runs only if the loop hits the round limit without breaking early.
        stop_reason = (
            f'Reached the automatic cap of {max_rounds} rounds before PlanningAgent returned FINISHED.'
        )

    final_report = planning_agent._complete(
        build_automatic_final_report_prompt(
            case_question=case_question,
            current_plan=current_plan,
            round_logs=round_logs,
            stop_reason=stop_reason,
        )
    )

    # Return one dictionary so the next cell can display a clean summary and
    # instructors can still inspect the raw traces if they want.
    return {
        'case_question': case_question,
        'planner_case_context': planner_case_context,
        'initial_plan': initial_plan,
        'round_logs': round_logs,
        'stop_reason': stop_reason,
        'final_plan': current_plan,
        'final_planner_decision': last_planner_decision,
        'final_report': final_report,
    }


# Helper function: format the automatic demo results as readable notebook Markdown.
def format_automatic_demo_result(result: dict) -> str:
    sections = [
        '### Automatic Demo Initial Plan\n\n' + result['initial_plan'],
    ]

    for log in result['round_logs']:
        sections.append(
            f"### Automatic Round {log['round_number']}\n\n"
            f"**PlanningAgent decision**\n\n{log['planner_decision']}\n\n"
            f"**Assigned task**\n\n`{log['task']}`\n\n"
            f"**ReactAgent final response**\n\n{log['react_response']}\n\n"
            f"**Revised plan after round {log['round_number']}**\n\n{log['revised_plan']}"
        )

    sections.append('### Automatic Loop Stop Reason\n\n' + result['stop_reason'])
    sections.append('### Automatic Demo Final Report\n\n' + result['final_report'])
    sections.append(
        "Raw React traces are stored in `auto_demo_result['round_logs'][i]['react_trace']` if you want to inspect them later."
    )
    return '\n\n'.join(sections)


### Step 4: Run the Automatic Multi-Round Demo

This cell runs the bounded loop with fresh state. It can stop early when `PlanningAgent` returns `FINISHED:`, when no usable next task is returned, or when the planner repeats the same task twice in a row.

What to look for after you run it:
- the initial plan that starts the loop
- each round's `PlanningAgent` decision and assigned task
- the `ReactAgent` evidence summary for that round
- the stop reason that explains why the loop ended
- the final report built from all collected observations

The main output shows readable round summaries and the final report. Raw `ReactAgent` traces stay inside `auto_demo_result['round_logs'][i]['react_trace']` if you want to inspect them later.


In [ ]:

# Run the bounded automatic demo and save the returned result dictionary.
auto_demo_result = run_automatic_planner_react_demo(max_rounds=AUTO_MAX_ROUNDS)

# Show the student-friendly Markdown summary instead of dumping the raw dictionary.
display(Markdown(format_automatic_demo_result(auto_demo_result)))


## Guided vs. Automatic Planner-to-`ReactAgent` Demos

Use `03b_lab_planner_react_workflow.ipynb` when students need the slower, step-by-step guided version.

Use this notebook when you want to demonstrate what a bounded automatic planning loop looks like in practice. It is closer to a realistic repeated workflow, but it is also more model-dependent and less predictable from run to run.

If the automatic output feels too dense, the most useful student questions are usually:
- What was the first task that `PlanningAgent` chose?
- What new `observation` changed the plan?
- Why did the loop stop?
- Which parts of the final report are clearly supported by evidence, and which parts remain uncertain?
